In [ ]:
import os
import sys
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# exp02_01 ルートをパスに追加
sys.path.append(os.path.abspath('..'))
from configs.config import *
from src.prompt_builder import build_prompt_df

# データ読み込み

`preprocess.ipynb` を先に実行してください。

In [ ]:
df_prep_train = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_train.pkl'))
df_prep_test = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_test.pkl'))
df_prep_udemy_activity = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_udemy_activity.pkl'))
df_prep_dx = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_dx.pkl'))
df_prep_hr = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_hr.pkl'))
df_prep_overtime_work_by_month = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_overtime_work_by_month.pkl'))
df_prep_position_history = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_position_history.pkl'))

print(f'train: {df_prep_train.shape}, test: {df_prep_test.shape}')

# プロンプト生成

In [ ]:
# ## trainプロンプト生成
df_prompt_train = build_prompt_df(
    df_base=df_prep_train[['社員番号', 'category']],
    df_overtime=df_prep_overtime_work_by_month,
    df_position=df_prep_position_history,
    df_dx=df_prep_dx,
    df_hr=df_prep_hr,
    df_udemy=df_prep_udemy_activity,
)
# labelsカラムを付与
df_prompt_train = df_prompt_train.merge(
    df_prep_train[['社員番号', 'category', 'target']],
    on=['社員番号', 'category'],
    how='left',
)
df_prompt_train = df_prompt_train.rename(columns={'target': 'labels'})
df_prompt_train['labels'] = df_prompt_train['labels'].astype(int)

print(f'train prompt shape: {df_prompt_train.shape}')
print(df_prompt_train.head(2))

In [ ]:
# ## プロンプト例を確認
print(df_prompt_train['prompt'].iloc[0])

In [ ]:
# ## testプロンプト生成
df_prompt_test = build_prompt_df(
    df_base=df_prep_test[['社員番号', 'category']],
    df_overtime=df_prep_overtime_work_by_month,
    df_position=df_prep_position_history,
    df_dx=df_prep_dx,
    df_hr=df_prep_hr,
    df_udemy=df_prep_udemy_activity,
)

print(f'test prompt shape: {df_prompt_test.shape}')

# トークン長の確認

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

list_token_lengths = [
    len(tokenizer.encode(p, truncation=False))
    for p in df_prompt_train['prompt']
]
df_token_len = pd.Series(list_token_lengths)
print('トークン長の統計:')
print(df_token_len.describe())
print(f'MAX_LENGTH={MAX_LENGTH} で切り捨てられる割合: {(df_token_len > MAX_LENGTH).mean():.2%}')

# データ保存

In [ ]:
os.makedirs(DIR_INTERIM, exist_ok=True)

df_prompt_train.to_pickle(os.path.join(DIR_INTERIM, 'df_prompt_exp02_01_train.pkl'))
df_prompt_test.to_pickle(os.path.join(DIR_INTERIM, 'df_prompt_exp02_01_test.pkl'))

print('プロンプトデータ保存完了')
print(f'  train: {os.path.join(DIR_INTERIM, "df_prompt_exp02_01_train.pkl")}')
print(f'  test:  {os.path.join(DIR_INTERIM, "df_prompt_exp02_01_test.pkl")}')